In [ ]:
# Import the libraries required for environment variables, JSON handling, website retrieval, and the OpenAI client.
import os
import json

from dotenv import load_dotenv
from scrape import fetch_website_contents
from openai import OpenAI

In [ ]:
# Load the OpenAI API key and initialize the OpenAI client.
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found, kindly recheck this.")
else:
    print("API key found.")

openai = OpenAI()

In [ ]:
# Define the content topic and research sources that will be used to create the content research brief.
content_topic = """
How AI automation is changing small business operations
"""

content_urls = [
    "https://www.ibm.com/think/topics/ai-automation",
    "https://www.microsoft.com/en-us/ai",
    "https://www.salesforce.com/artificial-intelligence/"
]

In [ ]:
# Collect and organize the content from each research source.
def collect_content_sources(urls):
    sources = []

    for url in urls:
        try:
            content = fetch_website_contents(url)

            sources.append({
                "url": url,
                "content": content
            })

        except Exception as error:
            print(f"Could not retrieve {url}: {error}")

    return sources


sources = collect_content_sources(content_urls)

print(f"Sources collected: {len(sources)}")

In [ ]:
# Define the instructions for turning research material into a structured content research brief.
content_prompt = """
You are a content research assistant.

Analyze the provided research sources and create a useful content research brief
for the given topic.

Requirements:

1. Identify the main findings relevant to the topic.
2. Extract important facts or ideas supported by the sources.
3. Identify useful content angles.
4. Identify questions that the content should answer.
5. Do not invent information.
6. Clearly distinguish information supported by the sources from suggestions.
7. Mention the source when presenting an important finding.

Return valid JSON using exactly this structure:

{
    "topic": "",
    "key_findings": [],
    "important_facts": [],
    "content_angles": [],
    "questions_to_answer": [],
    "sources": []
}

Return only valid JSON.
"""

In [ ]:
# Combine the collected sources into a single context for the content research assistant.
def build_content_context(sources):
    context = ""

    for index, source in enumerate(sources, start=1):
        context += f"""
SOURCE {index}
URL: {source['url']}

{source['content']}

--------------------
"""

    return context

In [ ]:
# Send the topic and research sources to the LLM and return the structured content research brief.
def research_content(topic, sources):
    context = build_content_context(sources)

    messages = [
        {"role": "system", "content": content_prompt},
        {
            "role": "user",
            "content": f"""
Content Topic:

{topic}

Research Sources:

{context}
"""
        }
    ]

    try:
        response = openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages
        )

        result = response.choices[0].message.content
        return json.loads(result)

    except json.JSONDecodeError:
        print("The LLM returned invalid JSON.")
        return None

    except Exception as error:
        print(f"An error occurred: {error}")
        return None

In [ ]:
# Run the content research automation and display the generated research brief.
result = research_content(content_topic, sources)

for field, value in result.items():
    print(f"\n{field.upper()}:")
    print(value)

In [ ]:
# Validate that the content research assistant returned all required fields.
content_fields = [
    "topic",
    "key_findings",
    "important_facts",
    "content_angles",
    "questions_to_answer",
    "sources"
]


def validate_content_research(data):
    if not data:
        return False

    return all(field in data for field in content_fields)


if validate_content_research(result):
    print("Content research is valid.")
else:
    print("Content research is missing required fields.")

In [ ]:
# Evaluate whether the generated content research brief contains the required research components.
required_sections = [
    "key_findings",
    "important_facts",
    "content_angles",
    "questions_to_answer",
    "sources"
]

completed_sections = []

for section in required_sections:
    if result.get(section):
        completed_sections.append(section)

coverage = len(completed_sections) / len(required_sections)

print(f"Completed sections: {completed_sections}")
print(f"Content Research Coverage: {coverage:.2%}")